# BoTorch Integration Tutorial: Synthetic Test Functions

This tutorial demonstrates how to use BoTorch's qExpectedImprovement (qEI) acquisition function with ALF on synthetic test functions. We'll optimize the Branin function, which has a known global optimum, allowing us to measure convergence objectively.

## Why Synthetic Test Functions?

Synthetic test functions provide several advantages for development and testing:
1. **Known ground truth**: We can measure true regret (distance from optimum)
2. **Fast evaluation**: No expensive experiments needed
3. **Domain agnostic**: Validates that the implementation works for any optimization problem
4. **Standard benchmarks**: Enables comparison with published results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from alf_core import BaseDatasetConfig, DesignTask, Modality, Optimizer, Oracle, Surrogate
from alf_tools.datasets.botorch_synthetic_dataset import BoTorchSyntheticDataset
from alf_tools.models.botorch_exact_gp_model import BoTorchGPModel
from alf_tools.optimizer.acquisition_functions.botorch_acquisition import BoTorchAcquisition
from alf_tools.optimizer.acquisition_functions.botorch_samplers import BoTorchMCSampler

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Setup: Create the Branin Test Function Dataset

The Branin function is a 2D function with 3 global minima. The known optimal value is -0.397887 (after negation for maximization).

In [ ]:
# Create dataset config
dataset_config = BaseDatasetConfig(
    name="branin",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.05,  # Start with 50 initial samples (5% of 1000)
    validation_frac=0.2,
    test_ratio=0.1,
    split_type="random",
)

# Create Branin dataset
dataset = BoTorchSyntheticDataset(
    config=dataset_config,
    function_name="branin",
    noise_std=0.0,  # Noiseless for this tutorial
    n_initial_samples=1000,
)

dataset.setup()
print(dataset)
print(f"\nTrue optimum: {dataset.true_optimum:.6f}")
print(f"Best in initial training set: {dataset.train_dataset.labels.max():.6f}")

## 2. Configure the Surrogate Model (Gaussian Process)

We'll use BoTorch's `BoTorchGPModel` which wraps SingleTaskGP with modern defaults, automatic input/output normalization, and gradient support for continuous optimization.

In [ ]:
# Create BoTorch GP model
# This model uses BoTorch's SingleTaskGP with modern defaults and
# automatic input/output normalization
gp_model = BoTorchGPModel(
    normalize_inputs=True,
    standardize_outputs=True,
    num_iterations=100,
    optimizer="scipy",  # Use scipy L-BFGS-B optimizer
)

# Wrap in Surrogate
surrogate = Surrogate(model=gp_model)

## 3. Configure BoTorch qEI Acquisition Function

BoTorch's qEI jointly optimizes batches of candidates for quality and diversity.

In [ ]:
# Get bounds for optimization
bounds_list = dataset.bounds.tolist()  # Convert to list of [lower, upper] for each dimension

# Create MC sampler for qEI
sampler = BoTorchMCSampler(sampler_type="sobol", num_samples=128, seed=42)

# Create BoTorch qEI acquisition function using the generic wrapper
acquisition_fn = BoTorchAcquisition(
    acquisition_type="qEI",  # Use qExpectedImprovement
    batch_size=5,  # Acquire 5 candidates per round
    bounds=bounds_list,
    num_restarts=10,
    raw_samples=512,
    sampler=sampler,
)

print(f"Acquisition function: BoTorch qEI with batch_size={acquisition_fn.batch_size}")
print(f"Bounds: {bounds_list}")

## 4. Setup Optimizer and Oracle

For continuous optimization with BoTorch, we use an empty search space (the acquisition function will optimize directly).

In [ ]:
# Create optimizer with continuous search for BoTorch
from alf_tools.optimizer.search.botorch_search_functions import ContinuousSearch

search_fn = ContinuousSearch()  # Returns empty list, signaling qEI to optimize directly

optimizer = Optimizer(
    acquisition_fn=acquisition_fn,
    search_fn=search_fn,
)

# Create oracle (uses dataset.query for evaluation)
oracle = Oracle(scorer=dataset)

## 5. Run Bayesian Optimization

We'll run 20 rounds of acquisition, acquiring 5 candidates per round (100 total evaluations).

In [ ]:
# Create design task
import shutil
from pathlib import Path

import pandas as pd
from alf_core.utils.task_state_logger import FileTaskStateLogger, TerminalTaskStateLogger

task = DesignTask(
    num_acq_rounds=20,
    acq_batch_size=5,
)

# Setup task
state = task.setup(
    dataset=dataset,
    surrogate=surrogate,
)

# Create loggers
terminal_logger = TerminalTaskStateLogger()
save_path = Path("results/gp_botorch_online_design/")
if save_path.exists():
    shutil.rmtree(save_path)
file_logger = FileTaskStateLogger(output_path=save_path)
loggers = [file_logger, terminal_logger]

# Run optimization
task.run(state=state, optimizer=optimizer, oracle=oracle, task_state_loggers=loggers)

# Load results from saved metrics
metrics_df = pd.read_csv(save_path / "metrics.csv")

print("\nOptimization complete!")
print(f"Best value found: {state.dataset.train_dataset.labels.max():.6f}")
print(f"True optimum: {dataset.true_optimum:.6f}")
print(f"Final regret: {dataset.true_optimum - state.dataset.train_dataset.labels.max():.6f}")

## 6. Analyze Results

Let's visualize the optimization progress with regret curves.

In [ ]:
# Extract metrics from the saved data
# Load acquisition round files to get best values over time
rounds = []
best_values = []
regrets = []

# Get the best value from training set at each round
for round_num in range(len(metrics_df)):
    if round_num == 0:  # Initial round
        continue
    rounds.append(round_num)

    # Load the acquisition file for this round to get all acquired candidates so far
    acq_file = save_path / f"acq_round_{round_num}.csv"
    if acq_file.exists():
        # Get cumulative best by reading all previous acquisition rounds
        all_labels = list(
            dataset.train_dataset.labels[: len(dataset.train_dataset)]
        )  # Initial training data
        for r in range(1, round_num + 1):
            acq_df = pd.read_csv(save_path / f"acq_round_{r}.csv")
            all_labels.extend(acq_df["label"].values)
        best_val = max(all_labels)
    else:
        # Fallback to current train dataset
        best_val = state.dataset.train_dataset.labels.max()

    best_values.append(best_val)
    regrets.append(dataset.true_optimum - best_val)

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Best value over rounds
ax1.plot(rounds, best_values, marker="o", linewidth=2, markersize=6)
ax1.axhline(y=dataset.true_optimum, color="r", linestyle="--", label="True optimum")
ax1.set_xlabel("Acquisition Round", fontsize=12)
ax1.set_ylabel("Best Value Found", fontsize=12)
ax1.set_title("Optimization Progress", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Regret over rounds (log scale)
ax2.semilogy(rounds, regrets, marker="o", linewidth=2, markersize=6, color="orange")
ax2.set_xlabel("Acquisition Round", fontsize=12)
ax2.set_ylabel("Regret (log scale)", fontsize=12)
ax2.set_title("Regret Convergence", fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal regret: {regrets[-1]:.6f}")
print(f"Improvement from initial: {(best_values[-1] - best_values[0]):.6f}")

## 6.5. Visualize Optimization Results

Now let's create a comprehensive visualization showing:
- **True Function**: The actual Branin function with known global optima
- **Learned Surrogate**: The GP model's predictions based on observed data
- **Model Uncertainty**: Where the GP is confident vs uncertain
- **Training Data Evolution**: Initial training points vs points acquired during optimization

In [ ]:
# Create a comprehensive visualization showing:
# - True Branin function
# - Training data (initial + acquired)
# - Learned surrogate model (GP predictions)
# - Acquired points throughout the run

# Create a dense grid for plotting
x1 = np.linspace(dataset.bounds[0, 0].item(), dataset.bounds[1, 0].item(), 100)
x2 = np.linspace(dataset.bounds[0, 1].item(), dataset.bounds[1, 1].item(), 100)
X1, X2 = np.meshgrid(x1, x2)
grid_points = np.column_stack([X1.ravel(), X2.ravel()])

# Evaluate true function on grid
true_values = dataset.test_fn(torch.tensor(grid_points, dtype=torch.float32))
true_values = true_values.numpy().reshape(X1.shape)

# Get surrogate predictions from the already-trained model
# Access the underlying BoTorch model directly for efficient grid prediction
grid_tensor = torch.tensor(grid_points, dtype=torch.float32)
surrogate.model.model.eval()
with torch.no_grad():
    posterior = surrogate.model.model.posterior(grid_tensor)
    mean_pred = posterior.mean.squeeze(-1).cpu().numpy().reshape(X1.shape)
    var_pred = posterior.variance.squeeze(-1).cpu().numpy().reshape(X1.shape)
    std_pred = np.sqrt(var_pred)

# Get training data from the final state
train_X = np.array([cand.data for cand in state.dataset.train_dataset.candidates])
train_y = state.dataset.train_dataset.labels

# Load all acquired points across rounds
acquired_X = []
acquired_y = []

for round_num in range(1, 21):  # 20 acquisition rounds
    acq_file = save_path / f"acq_round_{round_num}.csv"
    if acq_file.exists():
        acq_df = pd.read_csv(acq_file)
        acquired_y.extend(acq_df["label"].values)

# Convert lists to arrays
if len(acquired_y) > 0:
    # Reconstruct acquired_X from train_X (last N points where N = len(acquired_y))
    n_acquired = len(acquired_y)
    acquired_X_np = train_X[-n_acquired:]
    acquired_y_np = np.array(acquired_y)
else:
    acquired_X_np = np.array([]).reshape(0, 2)
    acquired_y_np = np.array([])

# Initial training data (first points)
initial_size = len(train_X) - len(acquired_y_np) if len(acquired_y_np) > 0 else len(train_X)
initial_X = train_X[:initial_size]
initial_y = train_y[:initial_size]

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. True Function
ax1 = axes[0, 0]
contour1 = ax1.contourf(X1, X2, true_values, levels=30, cmap="viridis")
ax1.set_title("True Branin Function", fontsize=14, fontweight="bold")
ax1.set_xlabel("x1", fontsize=12)
ax1.set_ylabel("x2", fontsize=12)
plt.colorbar(contour1, ax=ax1, label="Function Value")
# Mark the known optima
optima_x1 = [-np.pi, np.pi, 9.42478]
optima_x2 = [12.275, 2.275, 2.475]
ax1.scatter(
    optima_x1,
    optima_x2,
    c="red",
    s=200,
    marker="*",
    edgecolors="white",
    linewidths=2,
    label="True Optima",
    zorder=5,
)
ax1.legend(fontsize=10)

# 2. Surrogate Model (GP Mean)
ax2 = axes[0, 1]
contour2 = ax2.contourf(X1, X2, mean_pred, levels=30, cmap="viridis")
ax2.set_title("Learned Surrogate Model (GP Mean)", fontsize=14, fontweight="bold")
ax2.set_xlabel("x1", fontsize=12)
ax2.set_ylabel("x2", fontsize=12)
plt.colorbar(contour2, ax=ax2, label="Predicted Value")
# Plot training points
if len(initial_X) > 0:
    ax2.scatter(
        initial_X[:, 0],
        initial_X[:, 1],
        c="white",
        s=30,
        marker="o",
        edgecolors="black",
        linewidths=1,
        label=f"Initial Training ({len(initial_X)})",
        alpha=0.7,
        zorder=4,
    )
if len(acquired_X_np) > 0:
    ax2.scatter(
        acquired_X_np[:, 0],
        acquired_X_np[:, 1],
        c="red",
        s=80,
        marker="X",
        edgecolors="white",
        linewidths=2,
        label=f"Acquired Points ({len(acquired_X_np)})",
        zorder=5,
    )
ax2.legend(fontsize=10)

# 3. Surrogate Model Uncertainty (GP Std Dev)
ax3 = axes[1, 0]
contour3 = ax3.contourf(X1, X2, std_pred, levels=30, cmap="plasma")
ax3.set_title("Surrogate Model Uncertainty (GP Std Dev)", fontsize=14, fontweight="bold")
ax3.set_xlabel("x1", fontsize=12)
ax3.set_ylabel("x2", fontsize=12)
plt.colorbar(contour3, ax=ax3, label="Uncertainty (σ)")
# Plot training points
if len(initial_X) > 0:
    ax3.scatter(
        initial_X[:, 0],
        initial_X[:, 1],
        c="white",
        s=30,
        marker="o",
        edgecolors="black",
        linewidths=1,
        label=f"Initial Training ({len(initial_X)})",
        alpha=0.7,
        zorder=4,
    )
if len(acquired_X_np) > 0:
    ax3.scatter(
        acquired_X_np[:, 0],
        acquired_X_np[:, 1],
        c="yellow",
        s=80,
        marker="X",
        edgecolors="black",
        linewidths=2,
        label=f"Acquired Points ({len(acquired_X_np)})",
        zorder=5,
    )
ax3.legend(fontsize=10)

# 4. All Training Data with Values
ax4 = axes[1, 1]
contour4 = ax4.contourf(X1, X2, true_values, levels=30, cmap="viridis", alpha=0.3)
ax4.set_title("Training Data Evolution", fontsize=14, fontweight="bold")
ax4.set_xlabel("x1", fontsize=12)
ax4.set_ylabel("x2", fontsize=12)
# Plot initial training data
if len(initial_X) > 0:
    scatter1 = ax4.scatter(
        initial_X[:, 0],
        initial_X[:, 1],
        c=initial_y,
        s=50,
        cmap="coolwarm",
        marker="o",
        edgecolors="black",
        linewidths=1,
        label=f"Initial Training ({len(initial_X)})",
        alpha=0.8,
        zorder=3,
        vmin=train_y.min(),
        vmax=train_y.max(),
    )
# Plot acquired points with their actual values
if len(acquired_X_np) > 0:
    scatter2 = ax4.scatter(
        acquired_X_np[:, 0],
        acquired_X_np[:, 1],
        c=acquired_y_np,
        s=100,
        cmap="coolwarm",
        marker="X",
        edgecolors="white",
        linewidths=2,
        label=f"Acquired Points ({len(acquired_X_np)})",
        alpha=1.0,
        zorder=4,
        vmin=train_y.min(),
        vmax=train_y.max(),
    )
# Add colorbar for the scatter points
cbar = plt.colorbar(
    scatter2 if len(acquired_X_np) > 0 else scatter1, ax=ax4, label="Observed Value"
)
# Mark best point found
best_idx = np.argmax(train_y)
ax4.scatter(
    train_X[best_idx, 0],
    train_X[best_idx, 1],
    c="lime",
    s=300,
    marker="*",
    edgecolors="black",
    linewidths=2,
    label=f"Best Found ({train_y[best_idx]:.3f})",
    zorder=5,
)
ax4.legend(fontsize=10, loc="upper right")

plt.tight_layout()
plt.show()

print(f"\n{'=' * 60}")
print("Optimization Summary")
print(f"{'=' * 60}")
print(f"True optimum:         {dataset.true_optimum:.6f}")
print(f"Best value found:     {train_y.max():.6f}")
print(f"Final regret:         {dataset.true_optimum - train_y.max():.6f}")
print(f"Initial best:         {initial_y.max():.6f}")
print(f"Improvement:          {train_y.max() - initial_y.max():.6f}")
print(f"Total evaluations:    {len(train_X)}")
print(f"Initial training:     {len(initial_X)}")
print(f"Acquired points:      {len(acquired_X_np)}")
print(f"{'=' * 60}")

## 7. Comparison with Greedy Expected Improvement (Optional)

Let's compare BoTorch's qEI with a standard greedy EI approach.

In [ ]:
from alf_core.optimizer.search import DatasetSearch
from alf_tools.optimizer.acquisition_functions.expected_improvement import ExpectedImprovement

# Reset dataset
dataset_greedy = BoTorchSyntheticDataset(
    config=dataset_config,
    function_name="branin",
    noise_std=0.0,
    n_initial_samples=1000,
)
dataset_greedy.setup()

# Create greedy optimizer
greedy_acq = ExpectedImprovement()
search_fn_greedy = DatasetSearch()  # Searches candidate pool
optimizer_greedy = Optimizer(acquisition_fn=greedy_acq, search_fn=search_fn_greedy)
oracle_greedy = Oracle(scorer=dataset_greedy)

# Create new GP model for greedy approach
gp_model_greedy = BoTorchGPModel(
    normalize_inputs=True,
    standardize_outputs=True,
    num_iterations=100,
    optimizer="scipy",
)

# Run greedy optimization
task_greedy = DesignTask(num_acq_rounds=20, acq_batch_size=5)
state_greedy = task_greedy.setup(
    dataset=dataset_greedy,
    surrogate=Surrogate(model=gp_model_greedy),
)

# Create logger for greedy approach (optional, can skip to save time)
save_path_greedy = Path("results/gp_greedy_online_design/")
if save_path_greedy.exists():
    shutil.rmtree(save_path_greedy)
file_logger_greedy = FileTaskStateLogger(output_path=save_path_greedy)
terminal_logger_greedy = TerminalTaskStateLogger()

task_greedy.run(
    state=state_greedy,
    optimizer=optimizer_greedy,
    oracle=oracle_greedy,
    task_state_loggers=[file_logger_greedy, terminal_logger_greedy],
)

# Get final best values
best_value_greedy = state_greedy.dataset.train_dataset.labels.max()
best_value_qei = state.dataset.train_dataset.labels.max()

print(f"\nGreedy EI final best value: {best_value_greedy:.6f}")
print(f"BoTorch qEI final best value: {best_value_qei:.6f}")
print(f"Improvement of qEI over greedy: {(best_value_qei - best_value_greedy):.6f}")

## Summary

In this tutorial, we:
1. Created a synthetic Branin dataset using BoTorch test functions
2. Configured a Gaussian Process surrogate model
3. Used BoTorch's qEI for batch acquisition
4. Ran Bayesian optimization and measured convergence to the known optimum
5. Compared with greedy Expected Improvement

**Key Takeaways:**
- BoTorch's qEI jointly optimizes batches for quality and diversity
- Synthetic test functions enable rapid validation with known ground truth
- The same code works for any continuous optimization problem

**Next Steps:**
- Try other test functions (Hartmann, Ackley) to test on different landscapes
- Experiment with different batch sizes and MC samples
- Apply to real biological design problems (proteins, molecules)